# FAIR Organelle Segmentation pipeline

In [ ]:
import sys
sys.path.append('..')

import glob
import matplotlib.pyplot as plt
import numpy as np
import ome_zarr
from skimage.transform import resize
import torch
import torch.nn.functional

from src.fair_segmentation.image_util import *

## Load source data

In [ ]:
source_path = 'C:/Project/slides/AMC_EM/**/*.tif'

filenames = glob.glob(source_path)
filenames


## Load model

In [ ]:
# Mitonet models:
# https://zenodo.org/records/6327742
# https://zenodo.org/records/6453160

# https://github.com/volume-em/empanada-napari/blob/main/empanada_napari/configs/MitoNet_v1.yaml
# https://zenodo.org/record/6861565/files/MitoNet_v1.pth
# https://zenodo.org/record/6861565/files/MitoNet_v1_quantized.pth
model = torch.jit.load('../model/MitoNet_v1_quantized.pth')
model_ndim = 4  # Input shape must be `(N, C, H, W)`

## Process data

In [ ]:
downscale = 2

datas = []
for filename in filenames:
    data, metadata, pixel_size = extract_tiff_olympus(filename)
    new_shape = list(data.shape)
    new_shape[0] //= downscale
    new_shape[1] //= downscale
    data = resize(data, new_shape)
    datas.append(norm_image_variance2(data))

## Select data

In [ ]:
%matplotlib inline

data = datas[0]

plt.imshow(data)

## Run model

In [ ]:
while data.ndim < model_ndim:
    data = np.expand_dims(data, 0)
output = model(torch.from_numpy(data))
logits = output['sem_logits']
probabilities = torch.nn.functional.softmax(logits, dim=-1).detach().numpy()

## Show output

In [ ]:
%matplotlib inline

plt.imshow(probabilities[0][0])